In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [2]:
#Feature
def generate_batch(batch_size, S, n):
    input1 = torch.rand(batch_size, n)
    input2 = torch.rand(batch_size, n)
    dataset = input1 * torch.gt(input2, S).float()
    return dataset

In [3]:
# Neural Network Model
class NeuralNetwork(nn.Module):
    def __init__(self, n, m, activation):
        super(NeuralNetwork, self).__init__()
        self.n = n
        self.m = m
        self.weight = nn.init.xavier_normal_(torch.rand(m, n))
        self.bias = torch.zeros(n)
        self.activation = activation
        self.weight = nn.Parameter(self.weight)
        self.bias = nn.Parameter(self.bias)


    def forward(self, x):
        h = torch.matmul(x, self.weight.t())
        x_prime = self.activation(torch.matmul(h, self.weight) + self.bias)
        return x_prime
    

In [4]:
n = 5 # number of input features
m = 2 # number of hidden units
importance = torch.arange(0, n).float()  # importance of each feature
importance = torch.pow(0.7, importance)  # apply a power transformation to the importance values
def loss_fn (x, x_prime, importance):
    return torch.sum(importance * (x - x_prime) ** 2, dim=1).mean()
# m < n to ensure that the model is underparameterized
model = NeuralNetwork(n, m, nn.ReLU())
dataset = generate_batch(3, 0, n)

In [11]:
# Training loop model function
def train_model(n, batch_size, S, model, importance, epochs=7000, learning_rate=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
    for epoch in range(epochs):
        dataset = generate_batch(batch_size, S, n)
        optimizer.zero_grad()
        x_prime = model(dataset)
        loss = loss_fn(dataset, x_prime, importance)
        loss.backward()
        optimizer.step()
        if epoch % 100 == 0:
            print(f"Weights: {model.weight.data}")

In [ ]:
# Two features weight vector test to show superposition of features in the hidden layer
n = 2 # number of input features
m = 1 # number of hidden units
batch_size = 1000
importance = torch.ones(n).float()  # importance of each feature
S = 0.9 # threshold for generating the dataset
model_1 = NeuralNetwork(n, m, nn.ReLU())
train_model(n, batch_size, S, model_1, importance)

Weights: tensor([[-1.4252,  0.4628]])
Weights: tensor([[-1.3352,  0.5614]])
Weights: tensor([[-1.2705,  0.6577]])
Weights: tensor([[-1.2279,  0.7504]])
Weights: tensor([[-1.2011,  0.8318]])
Weights: tensor([[-1.1847,  0.8975]])
Weights: tensor([[-1.1732,  0.9438]])
Weights: tensor([[-1.1633,  0.9739]])
Weights: tensor([[-1.1551,  0.9899]])
Weights: tensor([[-1.1467,  0.9982]])
Weights: tensor([[-1.1371,  1.0029]])
Weights: tensor([[-1.1284,  1.0043]])
Weights: tensor([[-1.1182,  1.0053]])
Weights: tensor([[-1.1087,  1.0044]])
Weights: tensor([[-1.0983,  1.0047]])
Weights: tensor([[-1.0886,  1.0032]])
Weights: tensor([[-1.0775,  1.0042]])
Weights: tensor([[-1.0659,  1.0059]])
Weights: tensor([[-1.0550,  1.0061]])
Weights: tensor([[-1.0448,  1.0063]])
Weights: tensor([[-1.0356,  1.0050]])
Weights: tensor([[-1.0259,  1.0054]])
Weights: tensor([[-1.0172,  1.0054]])
Weights: tensor([[-1.0101,  1.0058]])
Weights: tensor([[-1.0072,  1.0057]])
Weights: tensor([[-1.0066,  1.0039]])
Weights: ten